# M5 Evaluation · Chronos-2 & TFT
Evaluate zero-shot / fine-tuned Chronos-2 and TFT forecasts against M5 actuals.

**Hardware target:** 2× Quadro RTX 5000 (16 GB each) · 20-core CPU · 62 GB RAM

Forecasts are cached as Parquet — re-running a cell reloads from disk unless
`force_run=True`.  The evaluator is initialised **once** and reused across all
experiments to avoid redundant scale/weight computation.

## 1 · Imports

In [1]:
%pip install nbformat

/home/nmwamsojo/tsfm-explo/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [10]:
import os
import gc
import json
import time
import warnings
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display

import sys; sys.path.append("../src/jobs/")  # adjust as needed to import local modules)
from m5_dataprep import M5DataPipeline
from m5_evaluator import M5Evaluator

from m5_exploration import (
    DEFAULT_CHRONOS_CONFIG,
)


# Both RTX 5000s exposed; GPU 0 has existing load (~359 MB).
# Route inference to GPU 1 (clean slate), evaluator GPU ops to GPU 0.
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'

INFERENCE_DEVICE = 'cuda:1' if torch.cuda.is_available() else 'cpu'
SCALE_GPU        = 'cuda:0'  # used by CuPy inside M5Evaluator if available

n_gpus = torch.cuda.device_count()
print(f'PyTorch sees {n_gpus} GPU(s)')
for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name}  {props.total_memory/1024**3:.1f} GB')

print(f'Inference device : {INFERENCE_DEVICE}')
print(f'CPU workers      : {os.cpu_count()} logical cores available')


PyTorch sees 2 GPU(s)
  GPU 0: Quadro RTX 5000  15.7 GB
  GPU 1: Quadro RTX 5000  15.7 GB
Inference device : cuda:1
CPU workers      : 20 logical cores available


## 2 · Paths & Experiment Config
Edit **only this cell** to define experiments. Everything below reads from these variables.

In [3]:
# ── Data paths ──────────────────────────────────────────────────────────
DATA_PATH     = '/mnt/lab/datasets/M5/jointed_M5.parquet'
CALENDAR_PATH = '/mnt/lab/nmwamsojo/m5_data/calendar.csv'
ACTUALS_PATH  = '/mnt/lab/nmwamsojo/m5_data/sales_test_evaluation.csv'

# Evaluation phase ends at d_1941; we forecast d_1942-d_1969 (28 days)
_CUTOFF_RAW = '2016-05-22'
CUTOFF_DAY  = (
    pd.to_datetime(_CUTOFF_RAW) - pd.Timedelta(days=28)
).strftime('%Y-%m-%d')
DATA_TAG = 'sales_only'  # used in model tags to denote which data prep was used
print(f'Evaluation cutoff : {CUTOFF_DAY}')

# ── Hyperparameter sweep grid ────────────────────────────────────────────
CONTEXT_LENGTHS = [2,4,8,16,32,64,128,256,512,1024]
BATCH_SIZES     = [64]

# ── AutoGluon wrapper settings ───────────────────────────────────────────
WRAPPER = {
    'eval_metric':          'RMSSE',
    'enable_ensemble':      False,
    'skip_model_selection': True,
    'verbosity':            1,
}

# ── Model configs (edit individual keys to override defaults) ────────────
CFG_CHRONOS_ZEROSHOT = {
    **DEFAULT_CHRONOS_CONFIG,
    'fine_tune_steps': 0,
    'known_cov_cols':  ['event_name_1', 'event_type_1'],
    'use_static':      False,
}

CFG_CHRONOS_FINETUNE = {
    **DEFAULT_CHRONOS_CONFIG,
    'fine_tune_steps': 500,
    'fine_tune_mode':  'lora',
    'fine_tune_lr':    1e-4,
    'known_cov_cols':  [],
    'use_static':      False,
}


# ---------------------------------------------------------------------------
# ── Build full experiment matrix (all context-length × batch-size combos) ─
# ---------------------------------------------------------------------------
EXPERIMENT_MATRIX = []
for dataprep_tag in ["sales_only", "sales_event_1", "sales_price"]:
    for clen in CONTEXT_LENGTHS:
        for bs in BATCH_SIZES:
            EXPERIMENT_MATRIX.append({
                'dataprep_tag': dataprep_tag,
                'model_tag':    f'hpo_cl_ens_fitAll_erratic_cl{clen}',
                'model_type':   'Chronos2',
                'config':       {**CFG_CHRONOS_ZEROSHOT, 'context_length': clen, 'batch_size': bs},
            })

# zeroshot + covariates
#for dataprep_tag in ['default', 'sales_event_1', "sales_price"]:
#    for clen in CONTEXT_LENGTHS:
#        for bs in BATCH_SIZES:
#            EXPERIMENT_MATRIX.append({
#                'dataprep_tag': dataprep_tag,
#                'model_tag':    f'hpo_chronos2_zeroshot_bs{bs}_cl{clen}',
#                'model_type':   'Chronos2',
#                'config':       {**CFG_CHRONOS_ZEROSHOT, 'context_length': clen, 'batch_size': bs},
#            })
#
#
#print(f'Experiments defined : {len(EXPERIMENT_MATRIX)}')


Evaluation cutoff : 2016-04-24


## 3 · Data Pipeline

In [4]:
dataprep_config = {"tag": DATA_TAG,
                   "base_cols": ['id', 'date', 'sales_quantity'],
                   "extra_cols": []
                }
pipeline = M5DataPipeline(config=dataprep_config)
hist_df, hist_df_trimmed, future_df, static_df, weights_scales = pipeline.get_prepared_data(DATA_PATH, CUTOFF_DAY, level=12) #, force_reprepare=True)

print(f"\nhist_df         : {hist_df.shape}  (columns: {list(hist_df.columns)})")
print(f"hist_df_trimmed : {hist_df_trimmed.shape}")
print(f"future_df       : {future_df.shape}  (columns: {list(future_df.columns)})")
print(f"static_df       : {static_df.shape}")
print(f"weights_scales  : {weights_scales.shape}  levels: {sorted(weights_scales['level'].unique())}")

del pipeline
gc.collect()

--- Cache Hit: Data found in /mnt/lab/nmwamsojo/prepared_data/sales_only/level_12/20160424 ---

hist_df         : (58327370, 18)  (columns: ['id', 'date', 'sales_quantity', 'wm_yr_wk', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'])
hist_df_trimmed : (45942500, 18)
future_df       : (853720, 17)  (columns: ['id', 'date', 'wm_yr_wk', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'])
static_df       : (30490, 6)
weights_scales  : (42840, 7)  levels: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]


20

## 4 · Load Actuals (done once)

In [5]:
def _wide_to_long(gt_wide: pd.DataFrame, calendar_path: str) -> pd.DataFrame:
    """
    Convert M5 evaluation CSV (wide, d_1942-d_1969 columns) to long format.
    Calendar join maps day codes (d_XXXX) to real calendar dates.
    """
    if 'id' not in gt_wide.columns:
        gt_wide['id'] = gt_wide['item_id'] + '_' + gt_wide['store_id'] + '_evaluation'
    day_cols = [c for c in gt_wide.columns if c.startswith('d_')]
    long = gt_wide.melt(id_vars=['id'], value_vars=day_cols,
                        var_name='d', value_name='sales_quantity')
    cal = pd.read_csv(calendar_path, usecols=['d', 'date'])
    cal['date'] = pd.to_datetime(cal['date'])
    long = long.merge(cal, on='d', how='left').drop(columns=['d'])
    long['id'] = long['id'].str.replace('_evaluation', '', regex=False)
    return long[['id', 'date', 'sales_quantity']]


# Evaluation window actuals (d_1942-d_1969)
_eval_raw      = pd.read_csv(ACTUALS_PATH)
df_actual_eval = _wide_to_long(_eval_raw, CALENDAR_PATH)
del _eval_raw
print(f'Eval actuals : {df_actual_eval.shape} | '
      f"{df_actual_eval['date'].min().date()} -> {df_actual_eval['date'].max().date()}")

# Full parquet actuals — evaluator uses these to compute WRMSSE
df_actual = (
    pd.read_parquet(DATA_PATH, columns=['id', 'date', 'sold'])
    .rename(columns={'sold': 'sales_quantity'})
)
df_actual['id'] = (
    df_actual['id'].astype(str)
    .str.replace('_evaluation', '', regex=False)
    .str.replace('_validation', '', regex=False)
)
print(f'Full actuals : {df_actual.shape}')


Eval actuals : (853720, 3) | 2016-05-23 -> 2016-06-19
Full actuals : (59181090, 3)


## 5 · Shared Evaluator (initialised once)
Pre-computes all 12-level scales and weights using the full training history.
**Reused across every experiment** — re-initialising per experiment wastes ~30 s
of redundant pivot and groupby work on the 30,490-series dataset.

In [6]:
def build_scale_smoothness_clusters(df_weights: pd.DataFrame,
                                    segments_df: pd.DataFrame) -> dict:
    # Merge once (avoid repeated joins later)
    df = df_weights.merge(
        segments_df[['id', 'smoothness_segs']].drop_duplicates(),
        on='id',
        how='inner'
    )

    # Define grouping logic
    dense = ["Smooth", "Erratic"]
    sparse = ["Intermittent", "Lumpy"]

    # Boolean masks (fast, vectorized)
    is_high = df["seller_type"] == "High"
    is_low  = df["seller_type"] == "Low"
    is_mhigh = df["seller_type"] == "Medium-High"
    is_mlow = df["seller_type"] == "Medium-Low"
    is_dense = df["smoothness_segs"].isin(dense)
    is_sparse = df["smoothness_segs"].isin(sparse)

    # Build clusters
    clusters = {
        "high_dense": list(df.loc[is_high & is_dense, "id"].unique()),
        "high_sparse": list(df.loc[is_high & is_sparse, "id"].unique()),
        "low_dense": list(df.loc[is_low & is_dense, "id"].unique()),
        "low_sparse": list(df.loc[is_low & is_sparse, "id"].unique()),
        "all": list(df["id"].unique())
    }
    # Build clusters
    clusters = {
        "high_dense": list(df_weights.loc[df_weights["seller_type"]=="High", "id"].unique()),
        "high_sparse": list(df_weights.loc[df_weights["seller_type"].isin(["Medium-High"]), "id"].unique()),
        "low_dense": list(df_weights.loc[df_weights["seller_type"].isin(["Medium-Low"]), "id"].unique()),
        "low_sparse": list(df_weights.loc[df_weights["seller_type"].isin(["Low"]), "id"].unique()),
        "all": list(df["id"].unique())
    }

    return clusters


In [11]:
_t0 = time.perf_counter()
# Trim each series to its first nonzero sale (used for L12 scales + all weights)

print(f'  Raw hist     : {hist_df.shape}')
print(f'  Trimmed hist : {hist_df_trimmed.shape}')

# M5Evaluator will parallelise scale computation across all CPU cores
# and optionally use CuPy on cuda:0 for the inner diff/mean ops.
evaluator = M5Evaluator(
    raw_train_df     = hist_df,
    trimmed_train_df = hist_df_trimmed,
    static_df        = static_df,
    target_col       = 'sales_quantity',
    price_col        = 'sell_price',
    use_gpu          = True,   # CuPy on cuda:0; silent noop if CuPy not installed
    n_jobs           = -1,     # all cores minus 2 headroom
)
print(f'Evaluator ready in {time.perf_counter()-_t0:.1f}s')



level = 12
info = evaluator.level_info[level]
df_weights = pd.DataFrame({
    'scale': info['scales'],
    'weight': info['weights']
}, index=info['index']).reset_index()



# Adding binning for analysis
weight_cuts = [0, 0.25,0.5,0.8, 1.0]    
df_weights['weight_segs'] = pd.qcut(df_weights['weight'], q=4, labels=['Low', 'Medium-Low', 'Medium-High', 'High'])




  Raw hist     : (58327370, 18)
  Trimmed hist : (45942500, 18)
  [CPU] CuPy not available — scale computation on 18 CPU cores.
  Building hierarchy scales and weights …
    Pivot [trimmed]: 30,490 series × 1,913 days
    Pivot [raw]: 30,490 series × 1,913 days
    Level  2 ready — 3 series
    Level  3 ready — 10 series
    Level  4 ready — 3 series
    Level  6 ready — 9 series
    Level  1 ready — 1 series
    Level  5 ready — 7 series
    Level  7 ready — 21 series
    Level  8 ready — 30 series
    Level  9 ready — 70 series
    Level 10 ready — 3,049 series
    Level 11 ready — 9,147 series
    Level 12 ready — 30,490 series
Evaluator ready in 85.3s


## 6 · Experiment Runner
`run_experiment` handles one (dataprep_tag, model_tag) pair: cache check →
forecast load → evaluation → metrics persist.  Designed to be called safely
from a `ThreadPoolExecutor` — `evaluate_all` is read-only after init.

In [12]:
from pathlib import Path

def get_forecast_paths(
    base_dir: str, 
    tag: str, 
    cutoff_day: str, 
    level: int, 
    model_tag: str
) -> dict:
    """
    Standalone path generator for M5 data and model artifacts.
    """
    # 1. Clean the date and establish the base directory for the cutoff
    date_str = cutoff_day.replace("-", "")
    base_folder = Path(base_dir) / tag / f"level_{level}" / date_str
    
    # 2. Define the specific model subdirectory
    model_folder = base_folder / "models" / model_tag
    
    # 3. Return a single dictionary containing all relevant paths
    return {
        "folder":    base_folder,
        "hist":      base_folder / "hist.parquet",
        "future":    base_folder / "future.parquet",
        "static":    base_folder / "static.parquet",
        "model_dir": model_folder,
        "forecast":  model_folder / "forecasts.parquet",
        "metrics":   model_folder / "metrics.json"
    }

## 7 · Parallel Batch Evaluation
All experiments with cached forecasts are evaluated concurrently.
`ThreadPoolExecutor` is used instead of processes: the bottleneck is
pandas I/O and numpy ops, both of which release the GIL.
With 20 cores and ~18 free we run up to 16 workers in parallel.
Peak RAM per worker ≈ 200-400 MB; 16 workers ≈ 6 GB — well within the 62 GB budget.

## 8 · Results Table

## 9 · Heatmap: Context Length x Batch Size

In [13]:
import os
import re
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _slugify(text: str, max_len: int = 200) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"[\s_-]+", "_", text)
    return text[:max_len].strip("_")


def plot_chronos_divergence(
    df: pd.DataFrame,
    title: str,
    out_dir: str = "/mnt/lab/nmwamsojo/plots",
    filename: str | None = None,
) -> str:
    """
    Deterministic plotting:
    - writes a single HTML file
    - safe for repeated overwrites
    - no rendering side-effects
    """

    # --- Extract context length ---
    if 'context_length' not in df.columns:
        df['context_length'] = df['model_tag'].str.extract(r'_cl(\d+)').astype(float)

    df = df.dropna(subset=['context_length', 'WRMSSE', 'WAPE_L12', 'RMSSE_L12'])
    df['context_length'] = df['context_length'].astype(int)

    # --- Aggregate ---
    summary = (
        df.groupby('context_length')[['WRMSSE', 'WAPE_L12', 'RMSSE_L12']]
        .mean()
        .sort_index()
        .reset_index()
    )

    # --- Normalize ---
    def normalize(s):
        d = s.max() - s.min()
        return (s - s.min()) / d if d > 0 else s * 0

    summary['WRMSSE_norm'] = normalize(summary['WRMSSE'])
    summary['RMSSE_norm']  = normalize(summary['RMSSE_L12'])
    summary['WAPE_norm']   = normalize(summary['WAPE_L12']) * 100

    # --- Plot ---
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(go.Scatter(
        x=summary['context_length'], y=summary['WRMSSE_norm'],
        name="WRMSSE", mode='lines+markers'
    ), secondary_y=False)

    fig.add_trace(go.Scatter(
        x=summary['context_length'], y=summary['RMSSE_norm'],
        name="RMSSE_L12", mode='lines+markers', line=dict(dash='dot')
    ), secondary_y=False)

    fig.add_trace(go.Scatter(
        x=summary['context_length'], y=summary['WAPE_norm'],
        name="WAPE", mode='lines+markers'
    ), secondary_y=True)

    fig.update_layout(
        title=title,
        template="plotly_white",
        xaxis_type="log"
    )

    # --- File handling ---
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    if filename is None:
        filename = _slugify(title) + ".html"

    final_path = os.path.join(out_dir, filename)
    tmp_path   = final_path + ".tmp"

    # Atomic write (prevents broken files during refresh)
    fig.write_html(tmp_path, include_plotlyjs="cdn")
    os.replace(tmp_path, final_path)

    print(f"[Plot saved] http://localhost:8050/{filename}")

    #return final_path

In [14]:
import os
import re
from pathlib import Path
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _slugify(text: str, max_len: int = 200) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s-]", "", text)
    text = re.sub(r"[\s_-]+", "_", text)
    return text[:max_len].strip("_")


# ── Visual constants ───────────────────────────────────────────────────────────
_C_WRMSSE  = "#2563eb"   # blue
_C_RMSSE   = "#16a34a"   # green
_C_WAPE    = "#dc2626"   # red

_FONT_FAMILY  = "Courier New, monospace"     # ticks — exact CL values read better in mono
_FONT_TITLE   = "Georgia, serif"
_FONT_LEGEND  = "Georgia, serif"

_SZ_TITLE     = 16
_SZ_AXIS_LBL  = 13
_SZ_TICK      = 11
_SZ_LEGEND    = 11

_LW_MAIN      = 2.5     # primary traces
_LW_DOT       = 2.0     # dotted trace
_MK_SIZE      = 8


def plot_chronos_divergence(
    df: pd.DataFrame,
    title: str,
    out_dir: str = "/mnt/lab/nmwamsojo/plots",
    filename: str | None = None,
) -> str:

    # --- Extract context length ---
    if "context_length" not in df.columns:
        df["context_length"] = df["model_tag"].str.extract(r"_cl(\d+)").astype(float)

    df = df.dropna(subset=["context_length", "WRMSSE", "WAPE_L12", "RMSSE_L12"])
    df["context_length"] = df["context_length"].astype(int)

    # --- Aggregate ---
    summary = (
        df.groupby("context_length")[["WRMSSE", "WAPE_L12", "RMSSE_L12"]]
        .mean()
        .sort_index()
        .reset_index()
    )

    # --- Normalize ---
    def normalize(s):
        d = s.max() - s.min()
        return (s - s.min()) / d if d > 0 else s * 0

    summary["WRMSSE_norm"] = normalize(summary["WRMSSE"])
    summary["RMSSE_norm"]  = normalize(summary["RMSSE_L12"])
    summary["WAPE_norm"]   = normalize(summary["WAPE_L12"]) * 100

    # --- X-axis: uniform index positions, real CL values as labels ----------
    # Replaces xaxis_type="log" which produces intermediate ticks (3,6,10…)
    # that collide with the actual power-of-2 context lengths.
    cl_vals = summary["context_length"].tolist()
    x_idx   = list(range(len(cl_vals)))          # 0,1,2 … evenly spaced
    x_labels = [str(v) for v in cl_vals]         # "2","4","8",…"1024"

    # --- Plot ---
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(go.Scatter(
        x=x_idx, y=summary["WRMSSE_norm"],
        name="WRMSSE (Aggregate)",
        mode="lines+markers",
        line=dict(color=_C_WRMSSE, width=_LW_MAIN),
        marker=dict(size=_MK_SIZE, symbol="circle",
                    color=_C_WRMSSE, line=dict(width=1.5, color="#ffffff")),
    ), secondary_y=False)

    fig.add_trace(go.Scatter(
        x=x_idx, y=summary["RMSSE_norm"],
        name="RMSSE_L12 (SKU)",
        mode="lines+markers",
        line=dict(color=_C_RMSSE, width=_LW_DOT, dash="dot"),
        marker=dict(size=_MK_SIZE, symbol="diamond",
                    color=_C_RMSSE, line=dict(width=1.5, color="#ffffff")),
    ), secondary_y=False)

    fig.add_trace(go.Scatter(
        x=x_idx, y=summary["WAPE_norm"],
        name="WAPE (Volume)",
        mode="lines+markers",
        line=dict(color=_C_WAPE, width=_LW_MAIN),
        marker=dict(size=_MK_SIZE, symbol="square",
                    color=_C_WAPE, line=dict(width=1.5, color="#ffffff")),
    ), secondary_y=True)

    # --- Shared axis style dicts ---
    _tick_font  = dict(family=_FONT_FAMILY, size=_SZ_TICK,  color="#374151")
    _label_font = dict(family=_FONT_TITLE,  size=_SZ_AXIS_LBL)

    fig.update_layout(
        title=dict(
            text=f"<b>{title}</b>",
            font=dict(family=_FONT_TITLE, size=_SZ_TITLE, color="#111827"),
            x=0.5, xanchor="center",
        ),
        template="plotly_white",
        paper_bgcolor="#ffffff",
        plot_bgcolor="#f9fafb",
        hovermode="x unified",
        margin=dict(l=72, r=72, t=80, b=64),
        legend=dict(
            orientation="h",
            y=1.06, x=0.5, xanchor="center",
            font=dict(family=_FONT_LEGEND, size=_SZ_LEGEND, color="#374151"),
            bgcolor="rgba(255,255,255,0.85)",
            bordercolor="#e5e7eb", borderwidth=1,
        ),
        # x-axis: uniform positions + real labels
        xaxis=dict(
            tickmode="array",
            tickvals=x_idx,
            ticktext=x_labels,
            tickangle=0,                          # keep labels horizontal
            tickfont=_tick_font,
            title=dict(
                text="Context Length (weeks — historical look-back)",
                font=dict(**_label_font, color="#6b7280"),
                standoff=12,
            ),
            showgrid=True,
            gridcolor="#e5e7eb",
            gridwidth=1,
            zeroline=False,
            linecolor="#d1d5db",
            linewidth=1,
            mirror=True,
        ),
    )

    # --- Left y-axis ---
    fig.update_yaxes(
        title_text="WRMSSE / RMSSE  (normalised, lower = better)",
        title_font=dict(**_label_font, color=_C_WRMSSE),
        title_standoff=12,
        tickfont=_tick_font,
        gridcolor="#e5e7eb",
        gridwidth=1,
        linecolor="#d1d5db",
        linewidth=1,
        mirror=True,
        secondary_y=False,
    )

    # --- Right y-axis ---
    fig.update_yaxes(
        title_text="WAPE  (normalised %, lower = better)",
        title_font=dict(**_label_font, color=_C_WAPE),
        title_standoff=12,
        tickfont=_tick_font,
        gridcolor="rgba(0,0,0,0)",   # suppress right-axis grid to avoid double grid
        linecolor="#d1d5db",
        linewidth=1,
        secondary_y=True,
    )

    # --- File handling ---
    Path(out_dir).mkdir(parents=True, exist_ok=True)

    if filename is None:
        filename = _slugify(title) + ".html"

    final_path = os.path.join(out_dir, filename)
    tmp_path   = final_path + ".tmp"

    fig.write_html(tmp_path, include_plotlyjs="cdn")
    os.replace(tmp_path, final_path)

    print(f"[Plot saved] http://localhost:8050/{filename}")

## Metrics by weight

In [15]:


import pandas as pd

# Extracting Level 12 info (Bottom level)

level = 12
info = evaluator.level_info[level]
df_weights = pd.DataFrame({
    'scale': info['scales'],
    'weight': info['weights']
}, index=info['index']).reset_index()

# Adding binning for analysis
df_weights['seller_type'] = pd.qcut(df_weights['weight'], 4, labels=['Low', 'Medium-Low', 'Medium-High', 'High'])
display(df_weights.sort_values('weight', ascending=False).head(10))

,id,scale,weight,seller_type
15906,HOBBIES_1_158_TX_3,73.051323,0.003642,High
17826,HOBBIES_1_354_TX_3,105.153343,0.003544,High
948,FOODS_1_096_WI_2,88.445084,0.002928,High
7322,FOODS_3_120_CA_3,780.646118,0.002494,High
17825,HOBBIES_1_354_TX_2,26.810488,0.001699,High
7320,FOODS_3_120_CA_1,822.633667,0.001423,High
17820,HOBBIES_1_354_CA_1,11.580261,0.001417,High
7468,FOODS_3_134_WI_2,74.824654,0.001402,High
7022,FOODS_3_090_CA_3,4929.767578,0.001319,High
946,FOODS_1_096_TX_3,95.323219,0.001309,High


In [ ]:
df_high_weights = df_weights[df_weights['seller_type'] == 'High'].sort_values('weight', ascending=False)
df_medium_low_weights = df_weights[df_weights['seller_type'] == 'Medium-Low'].sort_values('weight', ascending=False)

### High tier

In [17]:
def run_experiment_scoped(
    exp: dict,
    cutoff_day: str,
    shared_evaluator: M5Evaluator,
    actual_df: pd.DataFrame,
    selected_ids: list  # <--- New variable for limited scope
) -> dict | None:
    
    dtag      = exp['dataprep_tag']
    model_tag = exp['model_tag']

    pipeline   = M5DataPipeline({'model_tag': model_tag, 'tag': dtag})
    fcst_paths = pipeline.get_forecast_paths(cutoff_day, level=12)
    
    # Modify cache path so we don't overwrite full-catalog metrics
    #subset_metrics_path = fcst_paths['folder'] / "metrics_scoped_mlow.json"

   

    if not os.path.exists(fcst_paths['forecast']):
        return None

    # 1. Load the forecast
    fcst_df = pd.read_parquet(fcst_paths['forecast'])

    # 2. FILTERING LOGIC
    # Assuming the column name is 'id' or 'item_id' based on M5 standards
    # We filter both the forecast and the actuals to the selected scope
    scope_mask_fcst = fcst_df['id'].isin(selected_ids)
    scope_mask_act  = actual_df['id'].isin(selected_ids)
    
    scoped_fcst = fcst_df[scope_mask_fcst].copy()
    scoped_actual = actual_df[scope_mask_act].copy()

    if scoped_fcst.empty:
        print(f"  [WARN] No forecasts found for selected scope in {model_tag}")
        return None

    # 3. Evaluate only the scoped data
    metrics = shared_evaluator.evaluate_all(scoped_fcst, scoped_actual)

    # 4. Flatten results
    flat = {
        'dataprep_tag': dtag,
        'model_tag':    model_tag,
        'cutoff_day':   cutoff_day,
        'scope_size':   len(selected_ids),
        'WRMSSE':       metrics['WRMSSE'],
        'WAPE_L12':     metrics['WAPE_L12'],
        **metrics.get('level_scores', {}),
    }

    
    return flat

# --- Execution ---

# Define your limited scope here (e.g., specific high-value SKUs)

MY_SELECTED_IDS = df_high_weights.id.tolist()  # Top 100 by weight as an example


# Leave 2 cores for the OS and VSCode server processes visible in htop
MAX_WORKERS = min(16, os.cpu_count() - 2)
print(f'Parallel workers: {MAX_WORKERS}')

_t0  = time.perf_counter()
rows = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    future_to_exp = {
        pool.submit(
            run_experiment_scoped, 
            exp, 
            CUTOFF_DAY, 
            evaluator, 
            df_actual, 
            MY_SELECTED_IDS # <--- Pass the scope
        ): exp
        for exp in EXPERIMENT_MATRIX
    }

    done = 0
    for fut in as_completed(future_to_exp):
        exp  = future_to_exp[fut]
        done += 1
        try:
            result = fut.result()
        except Exception as exc:
            print(f"  [ERROR] {exp['model_tag']} -> {exc}")
            continue
        if result is None:
            continue
        rows.append(result)
        tag    = exp['model_tag']
        cached = 'cached' if result.get('_cached') else 'computed'
        print(f"  [{done:>3}/{len(EXPERIMENT_MATRIX)}] {tag}  "
              f"WRMSSE={result['WRMSSE']:.4f}  ({cached})")

print(f'\nBatch complete in {time.perf_counter()-_t0:.1f}s  ({len(rows)} results)')

if not rows:
    print('No results — check that forecast parquets exist on disk.')
else:
    level_cols   = [f'RMSSE_L{l}' for l in range(1, 13)]
    summary_cols = ['dataprep_tag', 'model_tag', 'WRMSSE', 'WAPE_L12', 'RMSE_L12', 'MAE_L12'] + level_cols

    df_results_high = (
        pd.DataFrame(rows)
        .drop(columns=['_cached', 'cutoff_day'], errors='ignore')
        .reindex(columns=summary_cols)
        .sort_values('WRMSSE')
        .reset_index(drop=True)
    )

    # Parse sweep dimensions from model tag for downstream filtering
    df_results_high['context_length'] = (
        df_results_high['model_tag'].str.extract(r'_cl(\d+)$').astype(float)
    )
    df_results_high['batch_size'] = (
        df_results_high['model_tag'].str.extract(r'_bs(\d+)_').astype(float)
    )

    display(
        df_results_high[['dataprep_tag', 'model_tag', 'WRMSSE', 'RMSSE_L12', 'WAPE_L12', 'context_length', 'batch_size']]
        .head(10)
        .style
        .format({'WRMSSE': '{:.4f}', 'RMSE_L12': '{:.4f}', 'WAPE_L12': '{:.4f}',
                 'context_length': '{:.0f}', 'batch_size': '{:.0f}'})
        .background_gradient(subset=['WRMSSE'], cmap='RdYlGn_r')
        .set_caption('Top-10 experiments by WRMSSE (lower is better)')
    )

    print(f"\nBest  WRMSSE = {df_results_high['WRMSSE'].min():.4f}  "
          f"({df_results_high.iloc[0]['model_tag']})")
    print(f"Worst WRMSSE = {df_results_high['WRMSSE'].max():.4f}")

plot_chronos_divergence(df_results_high, title='Chronos2 Zero-Shot Performance on High Weight Cluster')

Parallel workers: 16
  [ 21/30] hpo_cl_ens_fitAll_erratic_cl512  WRMSSE=0.7491  (computed)
  [ 22/30] hpo_cl_ens_fitAll_erratic_cl256  WRMSSE=0.7262  (computed)
  [ 23/30] hpo_cl_ens_fitAll_erratic_cl1024  WRMSSE=0.7415  (computed)
  [ 24/30] hpo_cl_ens_fitAll_erratic_cl4  WRMSSE=0.8760  (computed)
  [ 25/30] hpo_cl_ens_fitAll_erratic_cl8  WRMSSE=0.7673  (computed)
  [ 26/30] hpo_cl_ens_fitAll_erratic_cl32  WRMSSE=0.6475  (computed)
  [ 27/30] hpo_cl_ens_fitAll_erratic_cl2  WRMSSE=0.8463  (computed)
  [ 28/30] hpo_cl_ens_fitAll_erratic_cl16  WRMSSE=0.6555  (computed)
  [ 29/30] hpo_cl_ens_fitAll_erratic_cl128  WRMSSE=0.7174  (computed)
  [ 30/30] hpo_cl_ens_fitAll_erratic_cl64  WRMSSE=0.6526  (computed)

Batch complete in 88.8s  (10 results)


,dataprep_tag,model_tag,WRMSSE,RMSSE_L12,WAPE_L12,context_length,batch_size
0,sales_only,hpo_cl_ens_fitAll_erratic_cl32,0.6475,0.638329,0.5377,32,nan
1,sales_only,hpo_cl_ens_fitAll_erratic_cl64,0.6526,0.635941,0.5353,64,nan
2,sales_only,hpo_cl_ens_fitAll_erratic_cl16,0.6555,0.644389,0.5510,16,nan
3,sales_only,hpo_cl_ens_fitAll_erratic_cl128,0.7174,0.638524,0.5291,128,nan
4,sales_only,hpo_cl_ens_fitAll_erratic_cl256,0.7262,0.636912,0.5299,256,nan
5,sales_only,hpo_cl_ens_fitAll_erratic_cl1024,0.7415,0.636936,0.5288,1024,nan
6,sales_only,hpo_cl_ens_fitAll_erratic_cl512,0.7491,0.639142,0.5307,512,nan
7,sales_only,hpo_cl_ens_fitAll_erratic_cl8,0.7673,0.694566,0.5999,8,nan
8,sales_only,hpo_cl_ens_fitAll_erratic_cl2,0.8463,0.783229,0.6845,2,nan
9,sales_only,hpo_cl_ens_fitAll_erratic_cl4,0.8760,0.755263,0.6525,4,nan



Best  WRMSSE = 0.6475  (hpo_cl_ens_fitAll_erratic_cl32)
Worst WRMSSE = 0.8760
[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_high_weight_cluster.html


In [ ]:
plot_chronos_divergence(df_results_high[df_results_high["dataprep_tag"]=="sales_only"], title='Chronos2 Zero-Shot Performance on High Weight Cluster')

[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_high_weight_cluster.html


In [ ]:
df_mhigh_weights = df_weights[df_weights['seller_type'] == 'Medium-High'].sort_values('weight', ascending=False)

MY_SELECTED_IDS = df_mhigh_weights.id.tolist()  # Top 100 by weight as an example


# Leave 2 cores for the OS and VSCode server processes visible in htop
MAX_WORKERS = min(16, os.cpu_count() - 2)
print(f'Parallel workers: {MAX_WORKERS}')

_t0  = time.perf_counter()
rows = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    future_to_exp = {
        pool.submit(
            run_experiment_scoped, 
            exp, 
            CUTOFF_DAY, 
            evaluator, 
            df_actual, 
            MY_SELECTED_IDS # <--- Pass the scope
        ): exp
        for exp in EXPERIMENT_MATRIX
    }

    done = 0
    for fut in as_completed(future_to_exp):
        exp  = future_to_exp[fut]
        done += 1
        try:
            result = fut.result()
        except Exception as exc:
            print(f"  [ERROR] {exp['model_tag']} -> {exc}")
            continue
        if result is None:
            continue
        rows.append(result)
        tag    = exp['model_tag']
        cached = 'cached' if result.get('_cached') else 'computed'
        print(f"  [{done:>3}/{len(EXPERIMENT_MATRIX)}] {tag}  "
              f"WRMSSE={result['WRMSSE']:.4f}  ({cached})")

print(f'\nBatch complete in {time.perf_counter()-_t0:.1f}s  ({len(rows)} results)')

if not rows:
    print('No results — check that forecast parquets exist on disk.')
else:
    level_cols   = [f'RMSSE_L{l}' for l in range(1, 13)]
    summary_cols = ['dataprep_tag', 'model_tag', 'WRMSSE', 'WAPE_L12', 'RMSE_L12', 'MAE_L12'] + level_cols

    df_results_mhigh = (
        pd.DataFrame(rows)
        .drop(columns=['_cached', 'cutoff_day'], errors='ignore')
        .reindex(columns=summary_cols)
        .sort_values('WRMSSE')
        .reset_index(drop=True)
    )

    # Parse sweep dimensions from model tag for downstream filtering
    df_results_mhigh['context_length'] = (
        df_results_mhigh['model_tag'].str.extract(r'_cl(\d+)$').astype(float)
    )
    df_results_mhigh['batch_size'] = (
        df_results_mhigh['model_tag'].str.extract(r'_bs(\d+)_').astype(float)
    )

    display(
        df_results_mhigh[['dataprep_tag', 'model_tag', 'WRMSSE', 'RMSSE_L12', 'WAPE_L12', 'context_length', 'batch_size']]
        .head(10)
        .style
        .format({'WRMSSE': '{:.4f}', 'RMSE_L12': '{:.4f}', 'WAPE_L12': '{:.4f}',
                 'context_length': '{:.0f}', 'batch_size': '{:.0f}'})
        .background_gradient(subset=['WRMSSE'], cmap='RdYlGn_r')
        .set_caption('Top-10 experiments by WRMSSE (lower is better)')
    )

    print(f"\nBest  WRMSSE = {df_results_mhigh['WRMSSE'].min():.4f}  "
          f"({df_results_mhigh.iloc[0]['model_tag']})")
    print(f"Worst WRMSSE = {df_results_mhigh['WRMSSE'].max():.4f}")

plot_chronos_divergence(df_results_mhigh, title='Chronos2 Zero-Shot Performance on Medium High Weight Cluster')

Parallel workers: 16
  [  1/90] hpo_chronos2_zeroshot_bs256_cl2  WRMSSE=0.2742  (computed)
  [  2/90] hpo_chronos2_zeroshot_bs32_cl16  WRMSSE=0.3794  (computed)
  [  3/90] hpo_chronos2_zeroshot_bs32_cl8  WRMSSE=0.2708  (computed)
  [  4/90] hpo_chronos2_zeroshot_bs1024_cl4  WRMSSE=0.3087  (computed)
  [  5/90] hpo_chronos2_zeroshot_bs256_cl32  WRMSSE=0.5105  (computed)
  [  6/90] hpo_chronos2_zeroshot_bs1024_cl8  WRMSSE=0.2701  (computed)
  [  7/90] hpo_chronos2_zeroshot_bs1024_cl16  WRMSSE=0.3768  (computed)
  [  8/90] hpo_chronos2_zeroshot_bs256_cl8  WRMSSE=0.2702  (computed)
  [  9/90] hpo_chronos2_zeroshot_bs1024_cl32  WRMSSE=0.5062  (computed)
  [ 10/90] hpo_chronos2_zeroshot_bs32_cl4  WRMSSE=0.3119  (computed)
  [ 11/90] hpo_chronos2_zeroshot_bs256_cl4  WRMSSE=0.3090  (computed)
  [ 12/90] hpo_chronos2_zeroshot_bs256_cl16  WRMSSE=0.3787  (computed)
  [ 13/90] hpo_chronos2_zeroshot_bs32_cl2  WRMSSE=0.2740  (computed)
  [ 14/90] hpo_chronos2_zeroshot_bs32_cl64  WRMSSE=0.5699  (comp

,dataprep_tag,model_tag,WRMSSE,RMSSE_L12,WAPE_L12,context_length,batch_size
0,sales_event_1,hpo_chronos2_zeroshot_bs32_cl8,0.2549,0.155999,0.9416,8,32
1,sales_event_1,hpo_chronos2_zeroshot_bs256_cl8,0.2606,0.155459,0.9368,8,256
2,sales_event_1,hpo_chronos2_zeroshot_bs1024_cl8,0.2614,0.155460,0.9370,8,1024
3,sales_price,hpo_chronos2_zeroshot_bs1024_cl8,0.2641,0.155573,0.9238,8,1024
4,sales_price,hpo_chronos2_zeroshot_bs256_cl8,0.2650,0.155695,0.9242,8,256
5,sales_price,hpo_chronos2_zeroshot_bs32_cl2,0.2685,0.173076,1.0387,2,32
6,sales_price,hpo_chronos2_zeroshot_bs32_cl8,0.2688,0.156226,0.9247,8,32
7,sales_price,hpo_chronos2_zeroshot_bs256_cl2,0.2696,0.173204,1.0401,2,256
8,sales_only,hpo_chronos2_zeroshot_bs1024_cl8,0.2701,0.156085,0.9315,8,1024
9,sales_price,hpo_chronos2_zeroshot_bs1024_cl2,0.2702,0.173230,1.0406,2,1024



Best  WRMSSE = 0.2549  (hpo_chronos2_zeroshot_bs32_cl8)
Worst WRMSSE = 0.6292
[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_medium_high_weight_cluster.html


In [ ]:
df_mlow_weights = df_weights[df_weights['seller_type'] == 'Medium-Low'].sort_values('weight', ascending=False)

MY_SELECTED_IDS = df_mlow_weights.id.tolist()  # Top 100 by weight as an example


# Leave 2 cores for the OS and VSCode server processes visible in htop
MAX_WORKERS = min(16, os.cpu_count() - 2)
print(f'Parallel workers: {MAX_WORKERS}')

_t0  = time.perf_counter()
rows = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    future_to_exp = {
        pool.submit(
            run_experiment_scoped, 
            exp, 
            CUTOFF_DAY, 
            evaluator, 
            df_actual, 
            MY_SELECTED_IDS # <--- Pass the scope
        ): exp
        for exp in EXPERIMENT_MATRIX
    }

    done = 0
    for fut in as_completed(future_to_exp):
        exp  = future_to_exp[fut]
        done += 1
        try:
            result = fut.result()
        except Exception as exc:
            print(f"  [ERROR] {exp['model_tag']} -> {exc}")
            continue
        if result is None:
            continue
        rows.append(result)
        tag    = exp['model_tag']
        cached = 'cached' if result.get('_cached') else 'computed'
        print(f"  [{done:>3}/{len(EXPERIMENT_MATRIX)}] {tag}  "
              f"WRMSSE={result['WRMSSE']:.4f}  ({cached})")

print(f'\nBatch complete in {time.perf_counter()-_t0:.1f}s  ({len(rows)} results)')

if not rows:
    print('No results — check that forecast parquets exist on disk.')
else:
    level_cols   = [f'RMSSE_L{l}' for l in range(1, 13)]
    summary_cols = ['dataprep_tag', 'model_tag', 'WRMSSE', 'WAPE_L12', 'RMSE_L12', 'MAE_L12'] + level_cols

    df_results_mlow = (
        pd.DataFrame(rows)
        .drop(columns=['_cached', 'cutoff_day'], errors='ignore')
        .reindex(columns=summary_cols)
        .sort_values('WRMSSE')
        .reset_index(drop=True)
    )

    # Parse sweep dimensions from model tag for downstream filtering
    df_results_mlow['context_length'] = (
        df_results_mlow['model_tag'].str.extract(r'_cl(\d+)$').astype(float)
    )
    df_results_mlow['batch_size'] = (
        df_results_mlow['model_tag'].str.extract(r'_bs(\d+)_').astype(float)
    )

    display(
        df_results_mlow[['dataprep_tag', 'model_tag', 'WRMSSE', 'RMSSE_L12', 'WAPE_L12', 'context_length', 'batch_size']]
        .head(10)
        .style
        .format({'WRMSSE': '{:.4f}', 'RMSE_L12': '{:.4f}', 'WAPE_L12': '{:.4f}',
                 'context_length': '{:.0f}', 'batch_size': '{:.0f}'})
        .background_gradient(subset=['WRMSSE'], cmap='RdYlGn_r')
        .set_caption('Top-10 experiments by WRMSSE (lower is better)')
    )

    print(f"\nBest  WRMSSE = {df_results_mlow['WRMSSE'].min():.4f}  "
          f"({df_results_mlow.iloc[0]['model_tag']})")
    print(f"Worst WRMSSE = {df_results_mlow['WRMSSE'].max():.4f}")

plot_chronos_divergence(df_results_mlow, title='Chronos2 Zero-Shot Performance on Medium Low Weight Cluster')

Parallel workers: 16
  [  1/90] hpo_chronos2_zeroshot_bs32_cl32  WRMSSE=0.4595  (computed)
  [  2/90] hpo_chronos2_zeroshot_bs1024_cl32  WRMSSE=0.4566  (computed)
  [  3/90] hpo_chronos2_zeroshot_bs32_cl8  WRMSSE=0.2123  (computed)
  [  4/90] hpo_chronos2_zeroshot_bs1024_cl4  WRMSSE=0.1735  (computed)
  [  5/90] hpo_chronos2_zeroshot_bs256_cl32  WRMSSE=0.4584  (computed)
  [  6/90] hpo_chronos2_zeroshot_bs32_cl16  WRMSSE=0.3640  (computed)
  [  7/90] hpo_chronos2_zeroshot_bs32_cl4  WRMSSE=0.1753  (computed)
  [  8/90] hpo_chronos2_zeroshot_bs256_cl2  WRMSSE=0.1731  (computed)
  [  9/90] hpo_chronos2_zeroshot_bs32_cl2  WRMSSE=0.1736  (computed)
  [ 10/90] hpo_chronos2_zeroshot_bs1024_cl8  WRMSSE=0.2110  (computed)
  [ 11/90] hpo_chronos2_zeroshot_bs32_cl64  WRMSSE=0.4856  (computed)
  [ 12/90] hpo_chronos2_zeroshot_bs1024_cl16  WRMSSE=0.3609  (computed)
  [ 13/90] hpo_chronos2_zeroshot_bs256_cl16  WRMSSE=0.3627  (computed)
  [ 14/90] hpo_chronos2_zeroshot_bs1024_cl2  WRMSSE=0.1729  (com

,dataprep_tag,model_tag,WRMSSE,RMSSE_L12,WAPE_L12,context_length,batch_size
0,sales_event_1,hpo_chronos2_zeroshot_bs32_cl4,0.1569,0.074177,1.2136,4,32
1,sales_price,hpo_chronos2_zeroshot_bs1024_cl2,0.1604,0.075706,1.2043,2,1024
2,sales_price,hpo_chronos2_zeroshot_bs256_cl2,0.1611,0.075692,1.2036,2,256
3,sales_price,hpo_chronos2_zeroshot_bs32_cl2,0.1625,0.075608,1.2015,2,32
4,sales_price,hpo_chronos2_zeroshot_bs1024_cl4,0.1639,0.074473,1.2119,4,1024
5,sales_price,hpo_chronos2_zeroshot_bs256_cl4,0.1644,0.074545,1.2127,4,256
6,sales_event_1,hpo_chronos2_zeroshot_bs256_cl4,0.1650,0.073737,1.2041,4,256
7,sales_event_1,hpo_chronos2_zeroshot_bs1024_cl4,0.1661,0.073704,1.2035,4,1024
8,sales_price,hpo_chronos2_zeroshot_bs32_cl4,0.1676,0.074815,1.2148,4,32
9,sales_only,hpo_chronos2_zeroshot_bs1024_cl2,0.1729,0.076020,1.1996,2,1024



Best  WRMSSE = 0.1569  (hpo_chronos2_zeroshot_bs32_cl4)
Worst WRMSSE = 0.5085
[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_medium_low_weight_cluster.html


In [ ]:
plot_chronos_divergence(df_results_mlow[df_results_mlow["dataprep_tag"]=="sales_only"], title='Chronos2 Zero-Shot Performance on Medium Low Weight Cluster')

[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_medium_low_weight_cluster.html


In [ ]:
plot_chronos_divergence(df_results_low, title='Chronos2 Zero-Shot Performance on Low Weight Cluster')

[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_low_weight_cluster.html


In [ ]:
plot_chronos_divergence(df_results_mhigh, title='Chronos2 Zero-Shot Performance on Medium High Weight Cluster')

[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_medium_high_weight_cluster.html


In [ ]:
plot_chronos_divergence(df_results_high, title='Chronos2 Zero-Shot Performance on High Weight Cluster')

[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_high_weight_cluster.html


In [ ]:
plot_chronos_divergence(df_results_high[(df_results_high["dataprep_tag"]=="sales_only")&(df_results_high["batch_size"]==256)].sort_values("WRMSSE"), title='Chronos2 Zero-Shot Performance on High Weight 256 Cluster')

[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_high_weight_256_cluster.html


In [ ]:
plot_chronos_divergence(df_results_high[(df_results_high["dataprep_tag"]=="sales_only")&(df_results_high["batch_size"]==32)].sort_values("WRMSSE"), title='Chronos2 Zero-Shot Performance on High Weight 32 Cluster')

[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_high_weight_32_cluster.html


In [ ]:
plot_chronos_divergence(df_results_high[(df_results_high["dataprep_tag"]=="sales_price")&(df_results_high["batch_size"]==32)].sort_values("WRMSSE"), title='Chronos2 Zero-Shot Performance on High Weight 32 price Cluster')

[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_high_weight_32_price_cluster.html


In [ ]:
plot_chronos_divergence(df_results_high[(df_results_high["dataprep_tag"]=="sales_event_1")&(df_results_high["batch_size"]==32)].sort_values("WRMSSE"), title='Chronos2 Zero-Shot Performance on High Weight 32 event Cluster')

[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_high_weight_32_event_cluster.html


In [ ]:
def run_experiment_scoped(
    exp: dict,
    cutoff_day: str,
    shared_evaluator: M5Evaluator,
    actual_df: pd.DataFrame,
    selected_ids: list  # <--- New variable for limited scope
) -> dict | None:
    
    dtag      = exp['dataprep_tag']
    model_tag = exp['model_tag']

    pipeline   = M5DataPipeline({'model_tag': model_tag, 'tag': dtag})
    fcst_paths = pipeline.get_forecast_paths(cutoff_day, level=12)
    
    # Modify cache path so we don't overwrite full-catalog metrics
    #subset_metrics_path = fcst_paths['folder'] / "metrics_scoped_mlow.json"

   

    if not os.path.exists(fcst_paths['forecast']):
        return None

    # 1. Load the forecast
    fcst_df = pd.read_parquet(fcst_paths['forecast'])

    # 2. FILTERING LOGIC
    # Assuming the column name is 'id' or 'item_id' based on M5 standards
    # We filter both the forecast and the actuals to the selected scope
    scope_mask_fcst = fcst_df['id'].isin(selected_ids)
    scope_mask_act  = actual_df['id'].isin(selected_ids)
    
    scoped_fcst = fcst_df[scope_mask_fcst].copy()
    scoped_actual = actual_df[scope_mask_act].copy()

    if scoped_fcst.empty:
        print(f"  [WARN] No forecasts found for selected scope in {model_tag}")
        return None

    # 3. Evaluate only the scoped data
    metrics = shared_evaluator.evaluate_all(scoped_fcst, scoped_actual)

    # 4. Flatten results
    flat = {
        'dataprep_tag': dtag,
        'model_tag':    model_tag,
        'cutoff_day':   cutoff_day,
        'scope_size':   len(selected_ids),
        'WRMSSE':       metrics['WRMSSE'],
        'WAPE_L12':     metrics['WAPE_L12'],
        **metrics.get('level_scores', {}),
    }

    
    return flat

# --- Execution ---

# Define your limited scope here (e.g., specific high-value SKUs)

df_low_weights = df_weights[df_weights['seller_type'] == 'Low'].sort_values('weight', ascending=False)

MY_SELECTED_IDS = df_low_weights.id.tolist()  # Top 100 by weight as an example


# Leave 2 cores for the OS and VSCode server processes visible in htop
MAX_WORKERS = min(16, os.cpu_count() - 2)
print(f'Parallel workers: {MAX_WORKERS}')

_t0  = time.perf_counter()
rows = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    future_to_exp = {
        pool.submit(
            run_experiment_scoped, 
            exp, 
            CUTOFF_DAY, 
            evaluator, 
            df_actual, 
            MY_SELECTED_IDS # <--- Pass the scope
        ): exp
        for exp in EXPERIMENT_MATRIX
    }

    done = 0
    for fut in as_completed(future_to_exp):
        exp  = future_to_exp[fut]
        done += 1
        try:
            result = fut.result()
        except Exception as exc:
            print(f"  [ERROR] {exp['model_tag']} -> {exc}")
            continue
        if result is None:
            continue
        rows.append(result)
        tag    = exp['model_tag']
        cached = 'cached' if result.get('_cached') else 'computed'
        print(f"  [{done:>3}/{len(EXPERIMENT_MATRIX)}] {tag}  "
              f"WRMSSE={result['WRMSSE']:.4f}  ({cached})")

print(f'\nBatch complete in {time.perf_counter()-_t0:.1f}s  ({len(rows)} results)')

if not rows:
    print('No results — check that forecast parquets exist on disk.')
else:
    level_cols   = [f'RMSSE_L{l}' for l in range(1, 13)]
    summary_cols = ['dataprep_tag', 'model_tag', 'WRMSSE', 'WAPE_L12', 'RMSE_L12', 'MAE_L12'] + level_cols

    df_results_low = (
        pd.DataFrame(rows)
        .drop(columns=['_cached', 'cutoff_day'], errors='ignore')
        .reindex(columns=summary_cols)
        .sort_values('WRMSSE')
        .reset_index(drop=True)
    )

    # Parse sweep dimensions from model tag for downstream filtering
    df_results_low['context_length'] = (
        df_results_low['model_tag'].str.extract(r'_cl(\d+)$').astype(float)
    )
    df_results_low['batch_size'] = (
        df_results_low['model_tag'].str.extract(r'_bs(\d+)_').astype(float)
    )

    display(
        df_results_low[['dataprep_tag', 'model_tag', 'WRMSSE', 'RMSSE_L12', 'WAPE_L12', 'context_length', 'batch_size']]
        .head(10)
        .style
        .format({'WRMSSE': '{:.4f}', 'RMSE_L12': '{:.4f}', 'WAPE_L12': '{:.4f}',
                 'context_length': '{:.0f}', 'batch_size': '{:.0f}'})
        .background_gradient(subset=['WRMSSE'], cmap='RdYlGn_r')
        .set_caption('Top-10 experiments by WRMSSE (lower is better)')
    )

    print(f"\nBest  WRMSSE = {df_results_low['WRMSSE'].min():.4f}  "
          f"({df_results_low.iloc[0]['model_tag']})")
    print(f"Worst WRMSSE = {df_results_low['WRMSSE'].max():.4f}")

plot_chronos_divergence(df_results_low, title='Chronos2 Zero-Shot Performance on Low Weight Cluster')

Parallel workers: 16
  [  1/90] hpo_chronos2_zeroshot_bs1024_cl8  WRMSSE=0.2692  (computed)
  [  2/90] hpo_chronos2_zeroshot_bs256_cl32  WRMSSE=0.3435  (computed)
  [  3/90] hpo_chronos2_zeroshot_bs32_cl32  WRMSSE=0.3430  (computed)
  [  4/90] hpo_chronos2_zeroshot_bs32_cl16  WRMSSE=0.3301  (computed)
  [  5/90] hpo_chronos2_zeroshot_bs1024_cl16  WRMSSE=0.3300  (computed)
  [  6/90] hpo_chronos2_zeroshot_bs32_cl2  WRMSSE=0.2300  (computed)
  [  7/90] hpo_chronos2_zeroshot_bs1024_cl4  WRMSSE=0.2166  (computed)
  [  8/90] hpo_chronos2_zeroshot_bs1024_cl32  WRMSSE=0.3438  (computed)
  [  9/90] hpo_chronos2_zeroshot_bs256_cl8  WRMSSE=0.2697  (computed)
  [ 10/90] hpo_chronos2_zeroshot_bs256_cl4  WRMSSE=0.2169  (computed)
  [ 11/90] hpo_chronos2_zeroshot_bs32_cl4  WRMSSE=0.2172  (computed)
  [ 12/90] hpo_chronos2_zeroshot_bs32_cl8  WRMSSE=0.2702  (computed)
  [ 13/90] hpo_chronos2_zeroshot_bs256_cl2  WRMSSE=0.2293  (computed)
  [ 14/90] hpo_chronos2_zeroshot_bs1024_cl2  WRMSSE=0.2292  (comp

,dataprep_tag,model_tag,WRMSSE,RMSSE_L12,WAPE_L12,context_length,batch_size
0,sales_event_1,hpo_chronos2_zeroshot_bs32_cl4,0.2067,0.017630,1.2240,4,32
1,sales_event_1,hpo_chronos2_zeroshot_bs256_cl4,0.2112,0.017472,1.2131,4,256
2,sales_event_1,hpo_chronos2_zeroshot_bs1024_cl4,0.2115,0.017457,1.2125,4,1024
3,sales_price,hpo_chronos2_zeroshot_bs1024_cl4,0.2143,0.017665,1.2173,4,1024
4,sales_price,hpo_chronos2_zeroshot_bs256_cl4,0.2145,0.017685,1.2176,4,256
5,sales_price,hpo_chronos2_zeroshot_bs32_cl4,0.2155,0.017747,1.2193,4,32
6,sales_only,hpo_chronos2_zeroshot_bs1024_cl4,0.2166,0.017775,1.2205,4,1024
7,sales_only,hpo_chronos2_zeroshot_bs256_cl4,0.2169,0.017791,1.2206,4,256
8,sales_only,hpo_chronos2_zeroshot_bs32_cl4,0.2172,0.017847,1.2226,4,32
9,sales_price,hpo_chronos2_zeroshot_bs1024_cl2,0.2224,0.017711,1.1955,2,1024



Best  WRMSSE = 0.2067  (hpo_chronos2_zeroshot_bs32_cl4)
Worst WRMSSE = 0.3493
[Plot saved] http://localhost:8050/chronos2_zero_shot_performance_on_low_weight_cluster.html


## 10 · Per-Level RMSSE Breakdown (Best Model)

## 11 · Export Results